# EcoBot CDMX — Chatbot basado en reglas para la separación de residuos

**Proyecto Fase 1 · Desarrollo de Aplicaciones con Inteligencia Artificial (MBIA2602)**  
Universidad Tecmilenio · Maestría en Inteligencia Artificial  
Alumno: Nazir Enoch Rosas Salgado · Profesor: Mackenson Polche

Asistente que orienta a residentes de la Ciudad de México sobre cómo separar sus residuos según la Norma Ambiental NADF-024-AMBT-2013: en qué contenedor va cada residuo, qué días se recoge, cómo prepararlo y dónde llevar los residuos de manejo especial.

**Ejecución:** `Entorno de ejecución → Ejecutar todas`. La última celda abre el modo de conversación.

## 1. Entorno

In [1]:
import sys, platform
!pip -q install "nltk>=3.8"
import nltk
print('Python', platform.python_version())
print('NLTK', nltk.__version__)
assert sys.version_info >= (3, 10), 'Se requiere Python 3.10 o superior'

Python 3.13.17
NLTK 3.10.4


## 2. Base de conocimiento e intenciones
Se separan en JSON para que el conocimiento del dominio pueda editarse sin tocar el código del motor.

In [2]:
%%writefile base_conocimiento.json
{
  "metadatos": {
    "nombre": "Base de conocimiento EcoBot CDMX",
    "version": "1.0",
    "actualizada": "2026-10-25",
    "fuentes": [
      "Norma Ambiental NADF-024-AMBT-2013 (SEDEMA, Ciudad de México)",
      "Calendario 2026 de recolección diferenciada (Gobierno de la Ciudad de México, difusión en medios)",
      "Calendarios 2026 de Reciclatrón y Mercado de Trueque (SEDEMA)"
    ],
    "aviso": "Los días de recolección pueden variar por alcaldía y ruta; confirma con el servicio de limpia de tu colonia."
  },
  "categorias": {
    "organico": {
      "nombre": "Orgánicos",
      "color": "verde",
      "dias": "martes, jueves y sábado",
      "descripcion": "residuos biodegradables: restos de comida, cáscaras, cascarones, huesos, café, té y residuos de jardinería",
      "preparacion": "Escúrrelos bien y no los mezcles con bolsas, servilletas sucias ni otros materiales."
    },
    "reciclable": {
      "nombre": "Inorgánicos reciclables",
      "color": "gris",
      "dias": "lunes, miércoles, viernes y domingo",
      "descripcion": "materiales con potencial de reciclaje: papel y cartón limpios, vidrio, PET y otros plásticos rígidos, latas, metales, envases multicapa (tetrapak), ropa y textiles",
      "preparacion": "Entrégalos limpios y secos: enjuaga envases, aplasta botellas y cajas para que ocupen menos espacio."
    },
    "no_reciclable": {
      "nombre": "Inorgánicos no reciclables",
      "color": "naranja",
      "dias": "lunes, miércoles, viernes y domingo",
      "descripcion": "residuos de aprovechamiento limitado: residuos sanitarios, pañales, toallas sanitarias, papel higiénico, colillas, unicel, chicles, curitas, empaques metalizados y materiales contaminados",
      "preparacion": "Ciérralos en bolsa. Los residuos sanitarios conviene envolverlos para proteger al personal de limpia."
    },
    "manejo_especial": {
      "nombre": "Manejo especial y voluminosos",
      "color": "café",
      "dias": "no se entregan en la recolección ordinaria",
      "descripcion": "residuos que requieren un manejo distinto: electrónicos, pilas, aceite de cocina usado, medicamentos caducos, focos ahorradores, muebles, colchones, llantas y cascajo",
      "preparacion": "Llévalos a jornadas de acopio (Reciclatrón, Mercado de Trueque) o consulta con tu alcaldía la recolección de voluminosos."
    }
  },
  "residuos": [
    {
      "id": "cáscaras",
      "alias": [
        "cascara",
        "cascaras",
        "cascara de platano",
        "cascara de fruta",
        "cascaras de verdura"
      ],
      "categoria": "organico"
    },
    {
      "id": "restos de comida",
      "alias": [
        "restos de comida",
        "sobras",
        "comida",
        "comida echada a perder",
        "tortilla",
        "tortillas",
        "pan",
        "arroz"
      ],
      "categoria": "organico"
    },
    {
      "id": "cascarón de huevo",
      "alias": [
        "cascaron",
        "cascarones",
        "cascaron de huevo",
        "cascara de huevo"
      ],
      "categoria": "organico"
    },
    {
      "id": "huesos",
      "alias": [
        "hueso",
        "huesos",
        "huesos de pollo"
      ],
      "categoria": "organico"
    },
    {
      "id": "café y té",
      "alias": [
        "cafe",
        "posos de cafe",
        "filtro de cafe",
        "infusion",
        "manzanilla",
        "sobre de manzanilla"
      ],
      "categoria": "organico"
    },
    {
      "id": "residuos de jardinería",
      "alias": [
        "pasto",
        "hojas secas",
        "hojas",
        "ramas",
        "flores",
        "plantas",
        "poda"
      ],
      "categoria": "organico"
    },
    {
      "id": "frutas y verduras",
      "alias": [
        "fruta",
        "frutas",
        "verdura",
        "verduras",
        "platano",
        "manzana",
        "aguacate",
        "jitomate"
      ],
      "categoria": "organico"
    },
    {
      "id": "papel",
      "alias": [
        "papel",
        "hojas de papel",
        "cuaderno",
        "cuadernos",
        "periodico",
        "revista",
        "revistas",
        "folleto"
      ],
      "categoria": "reciclable",
      "condicionado": true
    },
    {
      "id": "cartón",
      "alias": [
        "carton",
        "caja de carton",
        "cajas",
        "caja",
        "huevera de carton"
      ],
      "categoria": "reciclable",
      "condicionado": true
    },
    {
      "id": "botellas de PET",
      "alias": [
        "pet",
        "botella de plastico",
        "botellas de plastico",
        "botella de agua",
        "botella de refresco",
        "botella"
      ],
      "categoria": "reciclable"
    },
    {
      "id": "envases de plástico rígido",
      "alias": [
        "envase de plastico",
        "envases de plastico",
        "garrafon",
        "bote de shampoo",
        "envase de yogurt",
        "tapa",
        "tapas",
        "tapitas"
      ],
      "categoria": "reciclable"
    },
    {
      "id": "vidrio",
      "alias": [
        "vidrio",
        "frasco",
        "frascos",
        "botella de vidrio",
        "botellas de vidrio",
        "envase de vidrio"
      ],
      "categoria": "reciclable"
    },
    {
      "id": "latas y metales",
      "alias": [
        "lata",
        "latas",
        "aluminio",
        "lata de refresco",
        "lata de atun",
        "metal",
        "fierro"
      ],
      "categoria": "reciclable"
    },
    {
      "id": "tetrapak",
      "alias": [
        "tetrapak",
        "tetra pak",
        "envase de leche",
        "caja de leche",
        "envase de jugo",
        "caja de jugo"
      ],
      "categoria": "reciclable"
    },
    {
      "id": "ropa y textiles",
      "alias": [
        "ropa",
        "textiles",
        "tela",
        "playera",
        "pantalon"
      ],
      "categoria": "reciclable"
    },
    {
      "id": "pañales",
      "alias": [
        "panal",
        "panales",
        "panal desechable"
      ],
      "categoria": "no_reciclable"
    },
    {
      "id": "toallas sanitarias",
      "alias": [
        "toalla sanitaria",
        "toallas sanitarias",
        "tampon",
        "tampones",
        "toallas femeninas"
      ],
      "categoria": "no_reciclable"
    },
    {
      "id": "papel higiénico",
      "alias": [
        "papel higienico",
        "papel de bano",
        "kleenex",
        "panuelos desechables"
      ],
      "categoria": "no_reciclable"
    },
    {
      "id": "colillas",
      "alias": [
        "colilla",
        "colillas",
        "colillas de cigarro",
        "cigarro",
        "cigarros"
      ],
      "categoria": "no_reciclable"
    },
    {
      "id": "unicel",
      "alias": [
        "unicel",
        "plato de unicel",
        "vaso de unicel",
        "charola de unicel"
      ],
      "categoria": "no_reciclable"
    },
    {
      "id": "chicles",
      "alias": [
        "chicle",
        "chicles",
        "goma de mascar"
      ],
      "categoria": "no_reciclable"
    },
    {
      "id": "curitas e hisopos",
      "alias": [
        "curita",
        "curitas",
        "hisopo",
        "hisopos",
        "cotonete",
        "cotonetes"
      ],
      "categoria": "no_reciclable"
    },
    {
      "id": "empaques metalizados",
      "alias": [
        "bolsa de frituras",
        "bolsas de papitas",
        "bolsa de papas",
        "empaque metalizado",
        "envoltura de galletas",
        "frituras",
        "papitas"
      ],
      "categoria": "no_reciclable"
    },
    {
      "id": "calzado",
      "alias": [
        "zapato",
        "zapatos",
        "tenis",
        "calzado",
        "chanclas"
      ],
      "categoria": "no_reciclable"
    },
    {
      "id": "artículos de escritura",
      "alias": [
        "pluma",
        "plumas",
        "boligrafo",
        "boligrafos",
        "plumon",
        "plumones",
        "lapiz",
        "lapices"
      ],
      "categoria": "no_reciclable"
    },
    {
      "id": "preservativos",
      "alias": [
        "condon",
        "condones",
        "preservativo",
        "preservativos"
      ],
      "categoria": "no_reciclable"
    },
    {
      "id": "pilas",
      "alias": [
        "pila",
        "pilas",
        "bateria",
        "baterias"
      ],
      "categoria": "manejo_especial",
      "destino": "Reciclatrón (se reciben pilas) o contenedores de acopio de pilas."
    },
    {
      "id": "electrónicos",
      "alias": [
        "celular",
        "celulares",
        "computadora",
        "laptop",
        "televisor",
        "tele",
        "impresora",
        "cargador",
        "cables",
        "electronicos",
        "aparatos electricos",
        "licuadora",
        "tablet",
        "audifonos"
      ],
      "categoria": "manejo_especial",
      "destino": "Reciclatrón de SEDEMA. Deben ir completos: no se reciben equipos desarmados, rotos o contaminados."
    },
    {
      "id": "aceite de cocina usado",
      "alias": [
        "aceite",
        "aceite de cocina",
        "aceite usado",
        "manteca"
      ],
      "categoria": "manejo_especial",
      "destino": "Mercado de Trueque. Guárdalo frío en un envase con tapa (un garrafón o botella) y nunca lo tires al drenaje."
    },
    {
      "id": "medicamentos caducos",
      "alias": [
        "medicina",
        "medicinas",
        "medicamento",
        "medicamentos",
        "pastillas",
        "jarabe"
      ],
      "categoria": "manejo_especial",
      "destino": "contenedores de medicamentos caducos que hay en muchas farmacias."
    },
    {
      "id": "focos ahorradores y lámparas",
      "alias": [
        "foco",
        "focos",
        "foco ahorrador",
        "lampara fluorescente",
        "lamparas",
        "tubo fluorescente"
      ],
      "categoria": "manejo_especial",
      "destino": "un programa de acopio especializado: el Reciclatrón no los recibe porque contienen mercurio."
    },
    {
      "id": "voluminosos",
      "alias": [
        "mueble",
        "muebles",
        "colchon",
        "colchones",
        "sillon",
        "refrigerador",
        "lavadora",
        "estufa",
        "llanta",
        "llantas"
      ],
      "categoria": "manejo_especial",
      "destino": "la oficina de limpia de tu alcaldía, que agenda la recolección de voluminosos."
    },
    {
      "id": "cascajo",
      "alias": [
        "cascajo",
        "escombro",
        "residuos de construccion",
        "tabique",
        "cemento"
      ],
      "categoria": "manejo_especial",
      "destino": "un sitio autorizado para residuos de construcción; el camión doméstico no lo recibe."
    }
  ],
  "modificadores_contaminacion": [
    "sucio",
    "sucia",
    "sucios",
    "sucias",
    "grasa",
    "grasoso",
    "grasosa",
    "mojado",
    "mojada",
    "manchado",
    "manchada",
    "embarrado",
    "pizza",
    "usado",
    "usada"
  ],
  "programas": {
    "reciclatron": {
      "nombre": "Reciclatrón",
      "descripcion": "Jornadas de SEDEMA para entregar residuos electrónicos, eléctricos y pilas. Hay jornadas itinerantes por alcaldía, un punto permanente y el Mega Reciclatrón en la UNAM.",
      "no_recibe": "focos ahorradores, lámparas fluorescentes, cableado público, módems y equipos desarmados, rotos o contaminados",
      "proxima": "29 y 30 de octubre en la UAM Azcapotzalco; 12 y 13 de noviembre en la Universidad Iberoamericana Santa Fe; 21 de noviembre en Canal Nacional, Puente de Patos (Coyoacán); y 26 y 27 de noviembre en la UAM Iztapalapa (calendario 2026 de SEDEMA)"
    },
    "mercado_trueque": {
      "nombre": "Mercado de Trueque",
      "descripcion": "Programa de SEDEMA donde cambias reciclables separados y limpios por puntos verdes canjeables por hortalizas, plantas y productos locales. Se pide llevar al menos 1 kg de material.",
      "recibe": "latas de aluminio y fierro, botellas de vidrio, PET y HDPE, papel, cartón, tetrapak, aceite de cocina usado y electrónicos",
      "proximas": "8 de noviembre de 2026 en el Bosque de San Juan de Aragón y 13 de diciembre de 2026 en el Parque Tezozómoc"
    }
  }
}

Writing base_conocimiento.json


In [3]:
%%writefile intenciones.json
{
  "intenciones": [
    {
      "tag": "saludo",
      "descripcion": "El usuario inicia la conversación.",
      "patrones": [
        "hola",
        "buenos dias",
        "buenas tardes",
        "buenas noches",
        "que tal",
        "hey",
        "saludos",
        "buen dia"
      ],
      "respuestas": [
        "¡Hola! Soy EcoBot CDMX. Te ayudo a separar tus residuos según la norma NADF-024. Pregúntame dónde va algo, qué día pasa el camión o dónde llevar electrónicos y aceite.",
        "¡Qué tal! Dime qué residuo quieres tirar y te digo en qué contenedor va y qué día se recoge."
      ]
    },
    {
      "tag": "despedida",
      "descripcion": "El usuario termina la conversación.",
      "patrones": [
        "adios",
        "bye",
        "hasta luego",
        "nos vemos",
        "chao",
        "hasta pronto",
        "me voy",
        "salir"
      ],
      "respuestas": [
        "¡Hasta luego! Recuerda: limpio, seco y separado. 🌱",
        "Nos vemos. Gracias por separar tus residuos."
      ]
    },
    {
      "tag": "agradecimiento",
      "descripcion": "El usuario agradece la respuesta.",
      "patrones": [
        "gracias",
        "te agradezco",
        "muy amable",
        "excelente",
        "perfecto",
        "muchas gracias",
        "grax",
        "thx",
        "ok gracias"
      ],
      "respuestas": [
        "¡Con gusto! ¿Tienes otro residuo que quieras separar?",
        "Para eso estoy. Si te surge otra duda, pregúntame."
      ]
    },
    {
      "tag": "ayuda",
      "descripcion": "El usuario pregunta qué puede hacer el asistente.",
      "patrones": [
        "ayuda",
        "que puedes hacer",
        "que sabes",
        "como funcionas",
        "opciones",
        "menu",
        "en que me ayudas",
        "no se que preguntar"
      ],
      "respuestas": [
        "Puedo ayudarte con:\n  • Dónde va un residuo (ej. «¿dónde tiro las pilas?»)\n  • Días de recolección (ej. «¿qué día pasan los orgánicos?»)\n  • Colores de los contenedores\n  • Cómo preparar los reciclables\n  • Reciclatrón y Mercado de Trueque\n  • Qué dice la norma NADF-024"
      ]
    },
    {
      "tag": "clasificar_residuo",
      "descripcion": "El usuario pregunta en qué categoría va un residuo. Requiere la entidad residuo.",
      "patrones": [
        "donde tiro",
        "donde va",
        "donde van",
        "donde pongo",
        "en que bote",
        "en que contenedor",
        "es reciclable",
        "se recicla",
        "que hago con",
        "como separo",
        "donde deposito",
        "donde echo",
        "tirar",
        "a que bote"
      ],
      "respuestas": []
    },
    {
      "tag": "dias_recoleccion",
      "descripcion": "El usuario pregunta qué días pasa la recolección, general o por categoría.",
      "patrones": [
        "que dia",
        "que dias",
        "cuando pasa",
        "cuando pasan",
        "horario",
        "recogen",
        "recoleccion",
        "camion de la basura",
        "pasa el camion",
        "dia toca"
      ],
      "respuestas": []
    },
    {
      "tag": "colores",
      "descripcion": "El usuario pregunta por el código de colores de los contenedores.",
      "patrones": [
        "color",
        "colores",
        "de que color",
        "codigo de colores",
        "que color",
        "color del contenedor"
      ],
      "respuestas": [
        "Código de colores recomendado en CDMX:\n  • Verde: orgánicos\n  • Gris: inorgánicos reciclables\n  • Naranja: inorgánicos no reciclables\n  • Café: manejo especial y voluminosos"
      ]
    },
    {
      "tag": "info_categoria",
      "descripcion": "El usuario pregunta qué incluye una categoría.",
      "patrones": [
        "que es",
        "que son",
        "que va en",
        "que entra en",
        "que se considera",
        "ejemplos de",
        "que incluye"
      ],
      "respuestas": []
    },
    {
      "tag": "preparacion",
      "descripcion": "El usuario pregunta cómo preparar sus residuos antes de entregarlos.",
      "patrones": [
        "lavar",
        "lavo",
        "enjuagar",
        "limpiar",
        "como preparo",
        "como entrego",
        "aplastar",
        "hay que lavar",
        "tengo que lavar",
        "quitar etiqueta"
      ],
      "respuestas": [
        "Regla general: limpio, seco y separado. Enjuaga envases, aplasta botellas y cajas, escurre los orgánicos y no mezcles categorías en la misma bolsa."
      ]
    },
    {
      "tag": "reciclatron",
      "descripcion": "El usuario pregunta por el Reciclatrón o puntos de acopio de electrónicos.",
      "patrones": [
        "reciclatron",
        "acopio",
        "centro de acopio",
        "donde llevo electronicos",
        "jornada de reciclaje"
      ],
      "respuestas": []
    },
    {
      "tag": "mercado_trueque",
      "descripcion": "El usuario pregunta por el Mercado de Trueque.",
      "patrones": [
        "trueque",
        "mercado de trueque",
        "intercambiar",
        "cambiar basura",
        "puntos verdes",
        "canjear"
      ],
      "respuestas": []
    },
    {
      "tag": "normativa",
      "descripcion": "El usuario pregunta por la norma, la obligatoriedad o por qué separar.",
      "patrones": [
        "norma",
        "nadf",
        "ley",
        "obligatorio",
        "multa",
        "sancion",
        "por que separar",
        "para que separar",
        "es obligatorio"
      ],
      "respuestas": [
        "La Norma Ambiental NADF-024-AMBT-2013 establece cómo separar, clasificar, recolectar y almacenar los residuos en la CDMX en cuatro grupos: orgánicos, inorgánicos reciclables, inorgánicos no reciclables y de manejo especial y voluminosos. Desde enero de 2026 la separación es obligatoria y hay días fijos de recolección. No se anunció una multa: si no separas, el camión no se lleva tus residuos. El objetivo es reducir lo que llega a rellenos sanitarios y aprovechar más material."
      ]
    }
  ],
  "fallback": [
    "No estoy seguro de haber entendido. Puedo decirte dónde va un residuo, qué día pasa el camión o dónde llevar electrónicos y aceite. ¿Me lo dices de otra forma?",
    "Esa pregunta está fuera de lo que sé por ahora. Escribe «ayuda» para ver lo que puedo hacer."
  ]
}

Writing intenciones.json


## 3. Motor del chatbot
Pipeline por turno: normalización → `word_tokenize` → stopwords de NLTK → `SnowballStemmer('spanish')` → detección de residuos con `ngrams` y `edit_distance` → puntuación de intenciones por reglas → política de diálogo con contexto.

In [4]:
%%writefile ecobot.py
"""
EcoBot CDMX — Chatbot basado en reglas para orientar la separación de residuos
en la Ciudad de México conforme a la Norma Ambiental NADF-024-AMBT-2013.

Proyecto Fase 1 · Desarrollo de Aplicaciones con Inteligencia Artificial (MBIA2602)
Autor: Nazir Enoch Rosas Salgado · Universidad Tecmilenio

Pipeline por turno:
    entrada -> normalización -> tokenización (NLTK) -> filtrado de stopwords (NLTK)
            -> stemming Snowball (NLTK) -> extracción de entidades (residuo / categoría)
            -> puntuación de intenciones por reglas -> política de diálogo con contexto
            -> respuesta
"""

from __future__ import annotations

import json
import re
import unicodedata
from dataclasses import dataclass, field
from pathlib import Path

import nltk
from nltk.metrics.distance import edit_distance
from nltk.stem import SnowballStemmer
from nltk.tokenize import word_tokenize
from nltk.util import ngrams

RAIZ = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()


# --------------------------------------------------------------------------- #
# Recursos NLTK
# --------------------------------------------------------------------------- #
def asegurar_recursos_nltk() -> None:
    """Descarga los recursos de NLTK que el bot necesita, solo si faltan."""
    for recurso, ruta in [("stopwords", "corpora/stopwords"), ("punkt_tab", "tokenizers/punkt_tab")]:
        try:
            nltk.data.find(ruta)
        except LookupError:
            nltk.download(recurso, quiet=True)


asegurar_recursos_nltk()
from nltk.corpus import stopwords  # noqa: E402  (se importa después de descargar)

# Palabras interrogativas y la negación se conservan: cambian la intención
# («¿dónde?», «¿cuándo?») o la categoría («no reciclable»).
CONSERVAR = {"no", "donde", "cuando", "que", "como", "cual"}
STOPWORDS = {w for w in stopwords.words("spanish")} - CONSERVAR
STEMMER = SnowballStemmer("spanish")


# --------------------------------------------------------------------------- #
# Preprocesamiento
# --------------------------------------------------------------------------- #
def quitar_acentos(texto: str) -> str:
    # La ñ se conserva aparte para no convertir «año» en «ano».
    texto = texto.replace("ñ", "\0")
    texto = "".join(c for c in unicodedata.normalize("NFD", texto) if unicodedata.category(c) != "Mn")
    return texto.replace("\0", "n")


def preprocesar(texto: str) -> list[str]:
    """Normaliza, tokeniza con NLTK, quita stopwords y aplica stemming."""
    texto = quitar_acentos(texto.lower())
    texto = re.sub(r"[^a-z0-9\s]", " ", texto)
    tokens = word_tokenize(texto, language="spanish")
    # Las interrogativas no se truncan: Snowball reduce «cómo» y «comida» a la misma raíz «com».
    return [t if t in CONSERVAR else STEMMER.stem(t) for t in tokens if t not in STOPWORDS and len(t) > 1]


def similares(a: str, b: str, prefijo: bool = True, min_edicion: int = 5) -> bool:
    """Compara dos raíces tolerando errores de una letra (distancia de edición de NLTK)
    y, solo para entidades, variaciones por prefijo («celul» / «celular»)."""
    if a == b:
        return True
    corto = min(len(a), len(b))
    if prefijo and corto >= 4 and (a.startswith(b) or b.startswith(a)):
        return True
    return corto >= min_edicion and edit_distance(a, b) <= 1


def peso(raiz: str) -> float:
    """Las interrogativas aportan menos evidencia que las palabras de contenido."""
    return 0.5 if raiz in CONSERVAR else 1.0


# --------------------------------------------------------------------------- #
# Estructuras
# --------------------------------------------------------------------------- #
@dataclass
class Contexto:
    """Memoria de corto plazo de la conversación."""
    ultima_categoria: str | None = None
    ultimo_residuo: str | None = None
    esperando_residuo: bool = False
    fallos_seguidos: int = 0


@dataclass
class Turno:
    entrada: str
    tokens: list[str]
    intencion: str
    puntaje: float
    entidades: list[str]
    categoria: str | None
    respuesta: str
    regla: str = ""


@dataclass
class EcoBot:
    ruta_kb: Path = RAIZ / "base_conocimiento.json"
    ruta_intenciones: Path = RAIZ / "intenciones.json"
    contexto: Contexto = field(default_factory=Contexto)
    historial: list[Turno] = field(default_factory=list)

    def __post_init__(self) -> None:
        self.kb = json.loads(Path(self.ruta_kb).read_text(encoding="utf-8"))
        datos = json.loads(Path(self.ruta_intenciones).read_text(encoding="utf-8"))
        self.intenciones = {i["tag"]: i for i in datos["intenciones"]}
        self.fallback = datos["fallback"]
        self._rotacion: dict[str, int] = {}

        # Patrones de intención preprocesados con el mismo pipeline que la entrada.
        self.patrones = {
            tag: [preprocesar(p) for p in i["patrones"]] for tag, i in self.intenciones.items()
        }
        # Alias de residuos preprocesados, ordenados del más largo al más corto
        # para que «botella de vidrio» gane sobre «botella».
        self.alias = []
        for r in self.kb["residuos"]:
            for a in r["alias"]:
                raices = preprocesar(a)
                if raices:
                    self.alias.append((raices, r))
        self.alias.sort(key=lambda x: -len(x[0]))
        self.contaminantes = {STEMMER.stem(quitar_acentos(m)) for m in self.kb["modificadores_contaminacion"]}

    # ------------------------------------------------------------------ #
    # Reconocimiento
    # ------------------------------------------------------------------ #
    def detectar_residuos(self, tokens: list[str]) -> list[dict]:
        """Busca residuos de la base de conocimiento con n-gramas (NLTK).
        Primera pasada: coincidencia exacta. Segunda: coincidencia aproximada."""
        usados: set[int] = set()
        hallados: list[dict] = []
        for exacto in (True, False):
            for raices, residuo in self.alias:
                n = len(raices)
                for i, grama in enumerate(ngrams(tokens, n)):
                    pos = set(range(i, i + n))
                    if pos & usados:
                        continue
                    # Un alias de una sola palabra exige raíces más largas para tolerar
                    # errores; así «mañana» (manan) no se confunde con «manzana» (manzan).
                    min_ed = 6 if n == 1 else 5
                    ok = all((g == r) if exacto else similares(g, r, min_edicion=min_ed)
                             for g, r in zip(grama, raices))
                    if ok:
                        usados |= pos
                        if residuo not in hallados:
                            hallados.append(residuo)
        return hallados

    @staticmethod
    def detectar_categoria(tokens: list[str]) -> str | None:
        t = " ".join(tokens)
        if re.search(r"\bno recicl", t):
            return "no_reciclable"
        if re.search(r"\brecicl(?!atron)", t):
            return "reciclable"
        if re.search(r"\borgan", t):
            return "organico"
        if re.search(r"\b(especial|volumin)", t):
            return "manejo_especial"
        return None

    def puntuar_intenciones(self, tokens: list[str]) -> tuple[str | None, float]:
        """Cada patrón suma el peso de sus raíces si TODAS aparecen en la entrada.
        Gana la intención con mayor puntaje; en empate, la que aparece primero."""
        conjunto = set(tokens)
        mejor, mejor_p = None, 0.0
        for tag, patrones in self.patrones.items():
            p = sum(sum(peso(x) for x in pt) for pt in patrones
                    if pt and all(any(similares(x, y, prefijo=False) for y in conjunto) for x in pt))
            if p > mejor_p:
                mejor, mejor_p = tag, p
        return mejor, mejor_p

    # ------------------------------------------------------------------ #
    # Respuestas
    # ------------------------------------------------------------------ #
    def _rotar(self, opciones: list[str], clave: str) -> str:
        i = self._rotacion.get(clave, 0)
        self._rotacion[clave] = i + 1
        return opciones[i % len(opciones)]

    def _cat(self, clave: str) -> dict:
        return self.kb["categorias"][clave]

    def respuesta_residuo(self, residuo: dict, tokens: list[str]) -> tuple[str, str]:
        cat_clave = residuo["categoria"]
        # Regla de contaminación: papel o cartón sucio deja de ser reciclable.
        if residuo.get("condicionado") and any(similares(t, m) for t in tokens for m in self.contaminantes):
            cat_clave = "no_reciclable"
            c = self._cat(cat_clave)
            texto = (f"Si el {residuo['id']} está sucio, mojado o con grasa ya no se puede reciclar: "
                     f"va en {c['nombre'].upper()} (contenedor {c['color']}). Se recoge {c['dias']}.")
            return texto, cat_clave
        c = self._cat(cat_clave)
        if cat_clave == "manejo_especial":
            texto = (f"{residuo['id'].capitalize()} es un residuo de MANEJO ESPECIAL (contenedor {c['color']}). "
                     f"No lo entregues al camión ordinario; llévalo a {residuo['destino']}")
        else:
            texto = (f"{residuo['id'].capitalize()} → {c['nombre'].upper()} (contenedor {c['color']}). "
                     f"Se recoge {c['dias']}. {c['preparacion']}")
        if residuo.get("condicionado"):
            texto += " Ojo: solo si está limpio y seco."
        return texto, cat_clave

    def respuesta_dias(self, cat: str | None) -> str:
        if cat:
            c = self._cat(cat)
            if cat == "manejo_especial":
                return f"Los residuos de {c['nombre'].lower()} {c['dias']}. {c['preparacion']}"
            return f"{c['nombre']} se recogen {c['dias']}. {self.kb['metadatos']['aviso']}"
        org, rec = self._cat("organico"), self._cat("reciclable")
        return (f"Calendario general de recolección en CDMX:\n"
                f"  • Orgánicos: {org['dias']}\n"
                f"  • Inorgánicos (reciclables y no reciclables): {rec['dias']}\n"
                f"  • Manejo especial y voluminosos: no van en la recolección ordinaria\n"
                f"{self.kb['metadatos']['aviso']}")

    def respuesta_categoria(self, cat: str) -> str:
        c = self._cat(cat)
        return (f"{c['nombre']} (contenedor {c['color']}): {c['descripcion']}. "
                f"Recolección: {c['dias']}.")

    def respuesta_programa(self, clave: str) -> str:
        p = self.kb["programas"][clave]
        if clave == "reciclatron":
            return (f"{p['nombre']}: {p['descripcion']} No recibe {p['no_recibe']}. "
                    f"Próximas jornadas: {p['proxima']}.")
        return (f"{p['nombre']}: {p['descripcion']} Recibe {p['recibe']}. "
                f"Próximas fechas: {p['proximas']}.")

    # ------------------------------------------------------------------ #
    # Política de diálogo
    # ------------------------------------------------------------------ #
    def responder(self, entrada: str) -> str:
        tokens = preprocesar(entrada)
        residuos = self.detectar_residuos(tokens)
        categoria = self.detectar_categoria(tokens) if not residuos else None
        intencion, puntaje = self.puntuar_intenciones(tokens)
        ctx = self.contexto

        # Si hay un residuo y la intención es débil o de clasificación, se clasifica.
        if residuos and intencion in (None, "clasificar_residuo", "info_categoria", "ayuda", "colores"):
            intencion = "clasificar_residuo"

        respuesta, regla, cat_resp = "", "", None

        if intencion == "dias_recoleccion":
            cat = residuos[0]["categoria"] if residuos else (categoria or ctx.ultima_categoria)
            respuesta = self.respuesta_dias(cat)
            regla = "días con categoría" if cat else "días generales"
            cat_resp = cat

        elif intencion == "clasificar_residuo" and residuos:
            partes = []
            for r in residuos:
                txt, cat_resp = self.respuesta_residuo(r, tokens)
                partes.append(txt)
            respuesta = "\n".join(partes)
            regla = "clasificación por base de conocimiento"
            ctx.ultimo_residuo = residuos[-1]["id"]

        elif intencion == "clasificar_residuo":
            respuesta = "¿Qué residuo quieres tirar? Dime el objeto, por ejemplo «cáscaras», «pilas» o «botella de vidrio»."
            regla = "falta entidad: se pide aclaración"
            ctx.esperando_residuo = True

        elif intencion == "preparacion":
            cat = residuos[0]["categoria"] if residuos else (categoria or ctx.ultima_categoria)
            base = self._rotar(self.intenciones["preparacion"]["respuestas"], "preparacion")
            respuesta = base + (f"\nPara {self._cat(cat)['nombre'].lower()}: {self._cat(cat)['preparacion']}" if cat else "")
            regla = "preparación"
            cat_resp = cat

        elif intencion == "reciclatron":
            respuesta, regla = self.respuesta_programa("reciclatron"), "programa"

        elif intencion == "mercado_trueque":
            respuesta, regla = self.respuesta_programa("mercado_trueque"), "programa"

        elif intencion == "info_categoria" and (categoria or ctx.ultima_categoria):
            cat_resp = categoria or ctx.ultima_categoria
            respuesta, regla = self.respuesta_categoria(cat_resp), "descripción de categoría"

        elif intencion is None and categoria:
            cat_resp = categoria
            respuesta, regla = self.respuesta_categoria(categoria), "categoría mencionada sin verbo"

        elif intencion == "colores" and (categoria or ctx.ultima_categoria):
            cat_resp = categoria or ctx.ultima_categoria
            c = self._cat(cat_resp)
            respuesta = f"El contenedor de {c['nombre'].lower()} es {c['color'].upper()}."
            regla = "color de una categoría"

        elif intencion in ("saludo", "despedida", "agradecimiento", "ayuda", "colores", "normativa"):
            respuesta = self._rotar(self.intenciones[intencion]["respuestas"], intencion)
            regla = "respuesta estática"

        elif residuos:
            # Residuo detectado pero intención no prevista: se prioriza clasificar.
            intencion = "clasificar_residuo"
            txt, cat_resp = self.respuesta_residuo(residuos[0], tokens)
            respuesta, regla = txt, "residuo sin intención explícita"

        else:
            intencion = "fallback"
            ctx.fallos_seguidos += 1
            if ctx.fallos_seguidos >= 2:
                respuesta = self.intenciones["ayuda"]["respuestas"][0]
                regla = "fallback repetido: se muestra menú"
            else:
                respuesta = self._rotar(self.fallback, "fallback")
                regla = "fallback"

        if intencion != "fallback":
            ctx.fallos_seguidos = 0
        if intencion == "clasificar_residuo" and residuos:
            ctx.esperando_residuo = False
        if cat_resp:
            ctx.ultima_categoria = cat_resp

        self.historial.append(Turno(entrada, tokens, intencion or "fallback", puntaje,
                                    [r["id"] for r in residuos], cat_resp, respuesta, regla))
        return respuesta

    def reiniciar(self) -> None:
        self.contexto = Contexto()
        self.historial = []


def conversar() -> None:
    """Modo interactivo para Colab o terminal."""
    bot = EcoBot()
    print("EcoBot CDMX — escribe «salir» para terminar.\n")
    while True:
        entrada = input("Tú: ").strip()
        if not entrada:
            continue
        print("EcoBot:", bot.responder(entrada), "\n")
        if bot.historial[-1].intencion == "despedida":
            break


if __name__ == "__main__":
    conversar()


Writing ecobot.py


## 4. Demostración del preprocesamiento con NLTK

In [5]:
from ecobot import EcoBot, preprocesar, quitar_acentos, STOPWORDS, STEMMER
from nltk.tokenize import word_tokenize

frase = '¿Dónde tiro la botela de bidrio y las pilas?'
import re
limpia = re.sub(r'[^a-z0-9\s]', ' ', quitar_acentos(frase.lower()))
tokens = word_tokenize(limpia, language='spanish')
print('1. Original      :', frase)
print('2. Tokens        :', tokens)
sin_sw = [t for t in tokens if t not in STOPWORDS]
print('3. Sin stopwords :', sin_sw)
print('4. Raíces        :', preprocesar(frase))

bot = EcoBot()
print('\nResiduos detectados:', [r['id'] for r in bot.detectar_residuos(preprocesar(frase))])
print('Intención        :', bot.puntuar_intenciones(preprocesar(frase)))

1. Original      : ¿Dónde tiro la botela de bidrio y las pilas?
2. Tokens        : ['donde', 'tiro', 'la', 'botela', 'de', 'bidrio', 'y', 'las', 'pilas']
3. Sin stopwords : ['donde', 'tiro', 'botela', 'bidrio', 'pilas']
4. Raíces        : ['donde', 'tir', 'botel', 'bidri', 'pil']

Residuos detectados: ['pilas', 'vidrio']
Intención        : ('clasificar_residuo', 2.5)


## 5. Pruebas funcionales (12 conversaciones)
Cada conversación arranca con contexto limpio. Se compara la intención y los residuos obtenidos contra los esperados y se guardan evidencias en `evidencias/`.

In [6]:
%%writefile pruebas.py
"""
Pruebas funcionales de EcoBot CDMX: 12 conversaciones distintas.

Cada turno declara la intención esperada y, cuando aplica, los residuos esperados.
El script ejecuta cada conversación con un bot nuevo (contexto limpio), compara lo
obtenido contra lo esperado y guarda evidencias en la carpeta evidencias/.
"""

import csv
import json
from datetime import datetime
from pathlib import Path

from ecobot import EcoBot, RAIZ

CONVERSACIONES = [
    {"id": "C01", "perfil": "Residente que separa por primera vez",
     "turnos": [("Hola", "saludo", []),
                ("¿Dónde tiro las cáscaras de plátano?", "clasificar_residuo", ["cáscaras"]),
                ("¿Y qué día pasan?", "dias_recoleccion", []),
                ("Gracias", "agradecimiento", [])]},
    {"id": "C02", "perfil": "Usuario con electrónicos y pilas",
     "turnos": [("Buenas tardes", "saludo", []),
                ("Tengo pilas viejas y un celular que ya no sirve, ¿qué hago con ellos?", "clasificar_residuo", ["pilas", "electrónicos"]),
                ("¿Cuándo es el próximo Reciclatrón?", "reciclatron", []),
                ("Adiós", "despedida", [])]},
    {"id": "C03", "perfil": "Duda sobre papel y cartón contaminados",
     "turnos": [("¿La caja de pizza con grasa es reciclable?", "clasificar_residuo", ["cartón"]),
                ("¿Y el periódico?", "clasificar_residuo", ["papel"]),
                ("¿De qué color es el contenedor de reciclables?", "colores", [])]},
    {"id": "C04", "perfil": "Consulta de calendario y categorías",
     "turnos": [("¿Qué días pasa el camión de la basura?", "dias_recoleccion", []),
                ("¿Qué va en los inorgánicos no reciclables?", "info_categoria", []),
                ("¿Los pañales?", "clasificar_residuo", ["pañales"])]},
    {"id": "C05", "perfil": "Aceite de cocina y preparación de envases",
     "turnos": [("¿Dónde tiro el aceite de cocina usado?", "clasificar_residuo", ["aceite de cocina usado"]),
                ("¿Qué es el Mercado de Trueque?", "mercado_trueque", []),
                ("¿Hay que lavar las botellas antes de tirarlas?", "preparacion", ["botellas de PET"])]},
    {"id": "C06", "perfil": "Usuario que escribe con errores ortográficos",
     "turnos": [("donde va la botela de bidrio", "clasificar_residuo", ["vidrio"]),
                ("y las latas de atun", "clasificar_residuo", ["latas y metales"]),
                ("ok grax", "agradecimiento", [])]},
    {"id": "C07", "perfil": "Consulta incompleta que requiere aclaración",
     "turnos": [("Quiero tirar algo", "clasificar_residuo", []),
                ("unas pilas", "clasificar_residuo", ["pilas"]),
                ("Gracias", "agradecimiento", [])]},
    {"id": "C08", "perfil": "Pregunta sobre la normativa",
     "turnos": [("¿Es obligatorio separar la basura?", "normativa", []),
                ("¿Por qué separar si al final la revuelven?", "normativa", []),
                ("Hasta luego", "despedida", [])]},
    {"id": "C09", "perfil": "Preguntas fuera de alcance",
     "turnos": [("¿Cómo va a estar el clima mañana?", "fallback", []),
                ("¿Quién ganó el partido de ayer?", "fallback", []),
                ("ayuda", "ayuda", [])]},
    {"id": "C10", "perfil": "Residuos voluminosos",
     "turnos": [("Tengo un colchón viejo, ¿cómo me deshago de él?", "clasificar_residuo", ["voluminosos"]),
                ("¿Y qué día pasan por los voluminosos?", "dias_recoleccion", [])]},
    {"id": "C11", "perfil": "Varios residuos en una sola pregunta",
     "turnos": [("¿Dónde van las medicinas caducas y los focos ahorradores?", "clasificar_residuo", ["medicamentos caducos", "focos ahorradores y lámparas"]),
                ("¿El Reciclatrón recibe focos?", "reciclatron", ["focos ahorradores y lámparas"]),
                ("Perfecto, gracias", "agradecimiento", [])]},
    {"id": "C12", "perfil": "Residuo no incluido en la base de conocimiento",
     "turnos": [("¿Dónde tiro una bolsa de plástico?", "clasificar_residuo", []),
                ("¿Y los residuos orgánicos qué incluyen?", "info_categoria", []),
                ("¿Se recicla el unicel?", "clasificar_residuo", ["unicel"])]},
]


def ejecutar() -> dict:
    carpeta = RAIZ / "evidencias"
    carpeta.mkdir(exist_ok=True)
    filas, transcripcion = [], []
    sello = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    transcripcion.append(f"EcoBot CDMX — Evidencias de ejecución ({sello})\n" + "=" * 72)

    for conv in CONVERSACIONES:
        bot = EcoBot()  # contexto limpio por conversación
        transcripcion.append(f"\n[{conv['id']}] {conv['perfil']}\n" + "-" * 72)
        for n, (entrada, esperada, ents_esp) in enumerate(conv["turnos"], 1):
            respuesta = bot.responder(entrada)
            t = bot.historial[-1]
            ok_int = t.intencion == esperada
            ok_ent = sorted(t.entidades) == sorted(ents_esp)
            filas.append({
                "conversacion": conv["id"], "turno": n, "entrada": entrada,
                "tokens": " ".join(t.tokens), "intencion_esperada": esperada,
                "intencion_obtenida": t.intencion, "entidades_esperadas": "|".join(ents_esp),
                "entidades_obtenidas": "|".join(t.entidades), "categoria": t.categoria or "",
                "regla": t.regla, "intencion_ok": ok_int, "entidad_ok": ok_ent,
                "respuesta": respuesta,
            })
            marca = "OK" if ok_int and ok_ent else "FALLA"
            transcripcion.append(f"Usuario: {entrada}")
            transcripcion.append(f"  ↳ tokens={t.tokens} | intención={t.intencion} | entidades={t.entidades} | regla={t.regla} [{marca}]")
            transcripcion.append("EcoBot: " + respuesta.replace("\n", "\n        ") + "\n")

    total = len(filas)
    resumen = {
        "fecha": sello,
        "conversaciones": len(CONVERSACIONES),
        "turnos": total,
        "intenciones_correctas": sum(f["intencion_ok"] for f in filas),
        "entidades_correctas": sum(f["entidad_ok"] for f in filas),
        "turnos_correctos": sum(f["intencion_ok"] and f["entidad_ok"] for f in filas),
        "conversaciones_sin_falla": sum(
            all(f["intencion_ok"] and f["entidad_ok"] for f in filas if f["conversacion"] == c["id"])
            for c in CONVERSACIONES),
    }
    resumen["exactitud_intencion"] = round(resumen["intenciones_correctas"] / total, 3)
    resumen["exactitud_turno"] = round(resumen["turnos_correctos"] / total, 3)

    with open(carpeta / "resultados.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(filas[0].keys()))
        w.writeheader()
        w.writerows(filas)
    (carpeta / "conversaciones.txt").write_text("\n".join(transcripcion), encoding="utf-8")
    (carpeta / "resumen.json").write_text(json.dumps(resumen, ensure_ascii=False, indent=2), encoding="utf-8")
    return {"resumen": resumen, "filas": filas, "transcripcion": "\n".join(transcripcion)}


if __name__ == "__main__":
    r = ejecutar()
    print(r["transcripcion"])
    print("\nResumen:", json.dumps(r["resumen"], ensure_ascii=False, indent=2))


Writing pruebas.py


In [7]:
!python pruebas.py

EcoBot CDMX — Evidencias de ejecución (2026-10-25 17:33:44)

[C01] Residente que separa por primera vez
------------------------------------------------------------------------
Usuario: Hola
  ↳ tokens=['hol'] | intención=saludo | entidades=[] | regla=respuesta estática [OK]
EcoBot: ¡Hola! Soy EcoBot CDMX. Te ayudo a separar tus residuos según la norma NADF-024. Pregúntame dónde va algo, qué día pasa el camión o dónde llevar electrónicos y aceite.

Usuario: ¿Dónde tiro las cáscaras de plátano?
  ↳ tokens=['donde', 'tir', 'casc', 'platan'] | intención=clasificar_residuo | entidades=['cáscaras'] | regla=clasificación por base de conocimiento [OK]
EcoBot: Cáscaras → ORGÁNICOS (contenedor verde). Se recoge martes, jueves y sábado. Escúrrelos bien y no los mezcles con bolsas, servilletas sucias ni otros materiales.

Usuario: ¿Y qué día pasan?
  ↳ tokens=['que', 'dia', 'pas'] | intención=dias_recoleccion | entidades=[] | regla=días con categoría [OK]
EcoBot: Orgánicos se recogen martes, juev

', 'sirv', 'que', 'hag'] | intención=clasificar_residuo | entidades=['pilas', 'electrónicos'] | regla=clasificación por base de conocimiento [OK]
EcoBot: Pilas es un residuo de MANEJO ESPECIAL (contenedor café). No lo entregues al camión ordinario; llévalo a Reciclatrón (se reciben pilas) o contenedores de acopio de pilas.
        Electrónicos es un residuo de MANEJO ESPECIAL (contenedor café). No lo entregues al camión ordinario; llévalo a Reciclatrón de SEDEMA. Deben ir completos: no se reciben equipos desarmados, rotos o contaminados.

Usuario: ¿Cuándo es el próximo Reciclatrón?
  ↳ tokens=['cuando', 'proxim', 'reciclatron'] | intención=reciclatron | entidades=[] | regla=programa [OK]
EcoBot: Reciclatrón: Jornadas de SEDEMA para entregar residuos electrónicos, eléctricos y pilas. Hay jornadas itinerantes por alcaldía, un punto permanente y el Mega Reciclatrón en la UNAM. No recibe focos ahorradores, lámparas fluorescentes, cableado público, módems y equipos desarmados, rotos o conta

## 6. Validación independiente (20 consultas no usadas para ajustar reglas)

In [8]:
%%writefile validacion.py
"""
Set de validación independiente: 20 consultas de un solo turno que NO se usaron
para ajustar las reglas. Se ejecuta una vez para estimar el desempeño fuera de
las conversaciones de prueba.
"""

import csv
import json

from ecobot import EcoBot, RAIZ

CASOS = [
    ("¿En qué bote van los huesos de pollo?", "clasificar_residuo", ["huesos"]),
    ("¿Dónde pongo el cartón de huevo?", "clasificar_residuo", ["cartón"]),
    ("las servilletas usadas a dónde van", "clasificar_residuo", []),
    ("¿Qué día recogen la basura inorgánica?", "dias_recoleccion", []),
    ("¿El vidrio roto se recicla?", "clasificar_residuo", ["vidrio"]),
    ("tengo una laptop vieja", "clasificar_residuo", ["electrónicos"]),
    ("¿Las colillas de cigarro en qué contenedor?", "clasificar_residuo", ["colillas"]),
    ("¿Qué hago con un refri que ya no sirve?", "clasificar_residuo", ["voluminosos"]),
    ("¿Puedo tirar el aceite por el fregadero?", "clasificar_residuo", ["aceite de cocina usado"]),
    ("¿A qué hora pasa el camión?", "dias_recoleccion", []),
    ("¿Dónde queda el próximo mercado de trueque?", "mercado_trueque", []),
    ("buen día, tengo una duda", "saludo", []),
    ("¿Qué contenedor es el naranja?", "colores", []),
    ("¿Las bolsas de papitas se reciclan?", "clasificar_residuo", ["empaques metalizados"]),
    ("¿Hay multa si no separo?", "normativa", []),
    ("¿Dónde tiro el cubrebocas?", "clasificar_residuo", []),
    ("¿Las cajas de leche son reciclables?", "clasificar_residuo", ["tetrapak"]),
    ("¿Qué onda con el reciclatrón, qué reciben?", "reciclatron", []),
    ("restos de jardín, ¿dónde van?", "clasificar_residuo", ["residuos de jardinería"]),
    ("¿Recomiéndame una película?", "fallback", []),
]

if __name__ == "__main__":
    filas = []
    for entrada, esperada, ents in CASOS:
        bot = EcoBot()
        resp = bot.responder(entrada)
        t = bot.historial[-1]
        filas.append({"entrada": entrada, "intencion_esperada": esperada, "intencion_obtenida": t.intencion,
                      "entidades_esperadas": "|".join(ents), "entidades_obtenidas": "|".join(t.entidades),
                      "ok": t.intencion == esperada and sorted(t.entidades) == sorted(ents),
                      "respuesta": resp})
    ok = sum(f["ok"] for f in filas)
    with open(RAIZ / "evidencias" / "validacion.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(filas[0].keys()))
        w.writeheader()
        w.writerows(filas)
    for f in filas:
        print(("OK   " if f["ok"] else "FALLA"), f["entrada"], "→", f["intencion_obtenida"], f["entidades_obtenidas"] or "")
        if not f["ok"]:
            print("      ", f["respuesta"][:160].replace("\n", " "))
    print(json.dumps({"casos": len(filas), "correctos": ok, "exactitud": round(ok / len(filas), 3)}, ensure_ascii=False))


Writing validacion.py


In [9]:
!python validacion.py

OK    ¿En qué bote van los huesos de pollo? → clasificar_residuo huesos
OK    ¿Dónde pongo el cartón de huevo? → clasificar_residuo cartón
OK    las servilletas usadas a dónde van → clasificar_residuo 
OK    ¿Qué día recogen la basura inorgánica? → dias_recoleccion 
OK    ¿El vidrio roto se recicla? → clasificar_residuo vidrio
OK    tengo una laptop vieja → clasificar_residuo electrónicos
OK    ¿Las colillas de cigarro en qué contenedor? → clasificar_residuo colillas
OK    ¿Qué hago con un refri que ya no sirve? → clasificar_residuo voluminosos
OK    ¿Puedo tirar el aceite por el fregadero? → clasificar_residuo aceite de cocina usado
OK    ¿A qué hora pasa el camión? → dias_recoleccion 
OK    ¿Dónde queda el próximo mercado de trueque? → mercado_trueque 
OK    buen día, tengo una duda → saludo 
FALLA ¿Qué contenedor es el naranja? → clasificar_residuo 
       ¿Qué residuo quieres tirar? Dime el objeto, por ejemplo «cáscaras», «pilas» o «botella de vidrio».
OK    ¿Las bolsas de papitas 

## 7. Conversación interactiva
Escribe `salir` para terminar.

In [10]:
import sys
from ecobot import conversar
if 'google.colab' in sys.modules:
    conversar()
else:
    print('Modo interactivo disponible al ejecutar en Google Colab.')

Modo interactivo disponible al ejecutar en Google Colab.
